### **Supplementary Figure S1: Experimental reproducibility**

Validates reproducibility across biological replicates by comparing Native-Native and IVT-IVT pairwise coverage correlations. Set the data path in the cell below before running.

**Samples:** Native RNA (3 replicates), IVT RNA (3 replicates) — all pairwise combinations within each type

**Reference:** *E. coli* K-12 (ENA|U00096|U00096.3)

### Figures in this notebook: Replicate Reproducibility

| Figure | Description |
|--------|-------------|
| **Fig. S1** | 2×3 hexbin panel — Native-Native pairwise (top row) and IVT-IVT pairwise (bottom row) |

All figures generated at three resolutions: per-base, 100 bp bins, 1 kb bins.

---

### Requirements

- **Python** 3.10+ with `matplotlib`, `pandas`, `numpy`, `scipy`
- **Read counts**: obtained with `samtools view -c -F 2308` (primary alignments only) — hardcoded in the read counts cell

### Input file layout

Place the notebook and the following files/folders in the same directory:

```
your_folder/
├── supp_fig_S1_technical_reprod.ipynb
└── coverages/
    ├── native/
    │   ├── k12_native_full_bc1.txt
    │   ├── k12_native_full_bc2.txt
    │   └── k12_native_full_bc3.txt
    └── ivt/
        ├── k12_ivt_full_bc1.txt
        ├── k12_ivt_full_bc2.txt
        └── k12_ivt_full_bc3.txt
```

Coverage files are tab-separated with columns `chrom`, `position`, `depth` (standard `samtools depth` output). Output figures are saved to `per_base/`, `binned_100bp/`, and `binned_1kb/` subfolders.

> **Note:** `DATA_DIR` resolves to the directory Jupyter was launched from, which is normally the notebook's folder. If files fail to load, set `DATA_DIR` explicitly to an absolute path.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.gridspec import GridSpec
from pathlib import Path

plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['figure.dpi'] = 300

# set to the directory containing coverages/
DATA_DIR = Path('.').resolve()
# DATA_DIR = Path('/absolute/path/to/your/data')

## Load data

In [ ]:
def load_coverage_file(filepath):
    df = pd.read_csv(filepath, sep='\t', header=None,
                     names=['chromosome', 'position', 'depth'])
    print(f"  {filepath.name}: {len(df):,} positions")
    return df

native1 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc1.txt')
native2 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc2.txt')
native3 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc3.txt')

ivt1 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc1.txt')
ivt2 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc2.txt')
ivt3 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc3.txt')

## CPM normalisation

In [ ]:
# read counts from samtools view -c -F 2308
read_counts = {
    'native1': 88366,
    'native2': 235984,
    'native3': 347917,
    'ivt1': 72626,
    'ivt2': 50142,
    'ivt3': 49260
}

def calculate_cpm(df, total_reads):
    df_copy = df.copy()
    df_copy['cpm'] = (df_copy['depth'] / total_reads) * 1_000_000
    return df_copy

native1 = calculate_cpm(native1, read_counts['native1'])
native2 = calculate_cpm(native2, read_counts['native2'])
native3 = calculate_cpm(native3, read_counts['native3'])
ivt1 = calculate_cpm(ivt1, read_counts['ivt1'])
ivt2 = calculate_cpm(ivt2, read_counts['ivt2'])
ivt3 = calculate_cpm(ivt3, read_counts['ivt3'])

## Binning

In [ ]:
def bin_coverage(df, bin_size):
    binned = df.copy()
    binned['bin_start'] = ((binned['position'] - 1) // bin_size) * bin_size + 1
    agg = binned.groupby('bin_start', sort=True).agg(
        cpm=('cpm', 'mean')
    ).reset_index()
    agg['position'] = agg['bin_start'] + (bin_size - 1) / 2.0
    return agg


def merge_binned_samples(df1, df2, bin_size, name1, name2):
    df1_binned = bin_coverage(df1, bin_size)
    df2_binned = bin_coverage(df2, bin_size)
    merged = pd.merge(
        df1_binned[['position', 'cpm']],
        df2_binned[['position', 'cpm']],
        on='position',
        suffixes=(f'_{name1}', f'_{name2}')
    )
    print(f"  {name1} vs {name2} ({bin_size}bp bins): {len(merged):,} bins")
    return merged

## Merge replicate pairs

In [ ]:
def merge_two_samples(df1, df2, name1, name2):
    merged = pd.merge(
        df1[['position', 'cpm']],
        df2[['position', 'cpm']],
        on='position',
        suffixes=(f'_{name1}', f'_{name2}')
    )
    print(f"  {name1} vs {name2}: {len(merged):,} positions")
    return merged

# per-base
native_12 = merge_two_samples(native1, native2, 'Native Rep 1', 'Native Rep 2')
native_13 = merge_two_samples(native1, native3, 'Native Rep 1', 'Native Rep 3')
native_23 = merge_two_samples(native2, native3, 'Native Rep 2', 'Native Rep 3')

ivt_12 = merge_two_samples(ivt1, ivt2, 'IVT Rep 1', 'IVT Rep 2')
ivt_13 = merge_two_samples(ivt1, ivt3, 'IVT Rep 1', 'IVT Rep 3')
ivt_23 = merge_two_samples(ivt2, ivt3, 'IVT Rep 2', 'IVT Rep 3')

# 100 bp bins
native_12_100bp = merge_binned_samples(native1, native2, 100, 'Native Rep 1', 'Native Rep 2')
native_13_100bp = merge_binned_samples(native1, native3, 100, 'Native Rep 1', 'Native Rep 3')
native_23_100bp = merge_binned_samples(native2, native3, 100, 'Native Rep 2', 'Native Rep 3')

ivt_12_100bp = merge_binned_samples(ivt1, ivt2, 100, 'IVT Rep 1', 'IVT Rep 2')
ivt_13_100bp = merge_binned_samples(ivt1, ivt3, 100, 'IVT Rep 1', 'IVT Rep 3')
ivt_23_100bp = merge_binned_samples(ivt2, ivt3, 100, 'IVT Rep 2', 'IVT Rep 3')

# 1 kb bins
native_12_1kb = merge_binned_samples(native1, native2, 1000, 'Native Rep 1', 'Native Rep 2')
native_13_1kb = merge_binned_samples(native1, native3, 1000, 'Native Rep 1', 'Native Rep 3')
native_23_1kb = merge_binned_samples(native2, native3, 1000, 'Native Rep 2', 'Native Rep 3')

ivt_12_1kb = merge_binned_samples(ivt1, ivt2, 1000, 'IVT Rep 1', 'IVT Rep 2')
ivt_13_1kb = merge_binned_samples(ivt1, ivt3, 1000, 'IVT Rep 1', 'IVT Rep 3')
ivt_23_1kb = merge_binned_samples(ivt2, ivt3, 1000, 'IVT Rep 2', 'IVT Rep 3')

## Correlations

In [ ]:
def calculate_replicate_correlation(merged_df, name1, name2):
    cov1 = merged_df[f'cpm_{name1}'].values
    cov2 = merged_df[f'cpm_{name2}'].values

    # only positions with coverage in both samples
    mask = (cov1 > 0) & (cov2 > 0)
    cov1_filtered = cov1[mask]
    cov2_filtered = cov2[mask]

    # log transform for Pearson — consistent with what the hexbin axes show
    cov1_log = np.log10(cov1_filtered + 1)
    cov2_log = np.log10(cov2_filtered + 1)

    spearman_r, spearman_p = stats.spearmanr(cov1_filtered, cov2_filtered)
    pearson_r, pearson_p = stats.pearsonr(cov1_log, cov2_log)

    n_total = len(cov1)
    n_both_nonzero = len(cov1_filtered)

    print(f"{name1} vs {name2}: Spearman \u03c1 = {spearman_r:.3f}, Pearson r = {pearson_r:.3f} "
          f"({n_both_nonzero:,} positions, {100*n_both_nonzero/n_total:.1f}%)")

    return {
        'comparison': f'{name1} vs {name2}',
        'n_positions': n_both_nonzero,
        'n_total': n_total,
        'pct_covered_both': 100*n_both_nonzero/n_total,
        'spearman_r': spearman_r,
        'spearman_p': spearman_p,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p
    }

print("Per-base — Native-Native:")
stats_n12 = calculate_replicate_correlation(native_12, 'Native Rep 1', 'Native Rep 2')
stats_n13 = calculate_replicate_correlation(native_13, 'Native Rep 1', 'Native Rep 3')
stats_n23 = calculate_replicate_correlation(native_23, 'Native Rep 2', 'Native Rep 3')

print("\nPer-base — IVT-IVT:")
stats_i12 = calculate_replicate_correlation(ivt_12, 'IVT Rep 1', 'IVT Rep 2')
stats_i13 = calculate_replicate_correlation(ivt_13, 'IVT Rep 1', 'IVT Rep 3')
stats_i23 = calculate_replicate_correlation(ivt_23, 'IVT Rep 2', 'IVT Rep 3')

print("\n100bp bins — Native-Native:")
stats_n12_100bp = calculate_replicate_correlation(native_12_100bp, 'Native Rep 1', 'Native Rep 2')
stats_n13_100bp = calculate_replicate_correlation(native_13_100bp, 'Native Rep 1', 'Native Rep 3')
stats_n23_100bp = calculate_replicate_correlation(native_23_100bp, 'Native Rep 2', 'Native Rep 3')

print("\n100bp bins — IVT-IVT:")
stats_i12_100bp = calculate_replicate_correlation(ivt_12_100bp, 'IVT Rep 1', 'IVT Rep 2')
stats_i13_100bp = calculate_replicate_correlation(ivt_13_100bp, 'IVT Rep 1', 'IVT Rep 3')
stats_i23_100bp = calculate_replicate_correlation(ivt_23_100bp, 'IVT Rep 2', 'IVT Rep 3')

print("\n1kb bins — Native-Native:")
stats_n12_1kb = calculate_replicate_correlation(native_12_1kb, 'Native Rep 1', 'Native Rep 2')
stats_n13_1kb = calculate_replicate_correlation(native_13_1kb, 'Native Rep 1', 'Native Rep 3')
stats_n23_1kb = calculate_replicate_correlation(native_23_1kb, 'Native Rep 2', 'Native Rep 3')

print("\n1kb bins — IVT-IVT:")
stats_i12_1kb = calculate_replicate_correlation(ivt_12_1kb, 'IVT Rep 1', 'IVT Rep 2')
stats_i13_1kb = calculate_replicate_correlation(ivt_13_1kb, 'IVT Rep 1', 'IVT Rep 3')
stats_i23_1kb = calculate_replicate_correlation(ivt_23_1kb, 'IVT Rep 2', 'IVT Rep 3')

In [ ]:
datasets = {
    'per_base': {
        'native_data': [
            (native_12, stats_n12, 'Native Rep 1', 'Native Rep 2', 'Native Rep1 vs Rep2'),
            (native_13, stats_n13, 'Native Rep 1', 'Native Rep 3', 'Native Rep1 vs Rep3'),
            (native_23, stats_n23, 'Native Rep 2', 'Native Rep 3', 'Native Rep2 vs Rep3')
        ],
        'ivt_data': [
            (ivt_12, stats_i12, 'IVT Rep 1', 'IVT Rep 2', 'IVT Rep1 vs Rep2'),
            (ivt_13, stats_i13, 'IVT Rep 1', 'IVT Rep 3', 'IVT Rep1 vs Rep3'),
            (ivt_23, stats_i23, 'IVT Rep 2', 'IVT Rep 3', 'IVT Rep2 vs Rep3')
        ],
        'stats_list': [stats_n12, stats_n13, stats_n23, stats_i12, stats_i13, stats_i23],
        'suffix': '',
        'label': 'Per-base'
    },
    'binned_100bp': {
        'native_data': [
            (native_12_100bp, stats_n12_100bp, 'Native Rep 1', 'Native Rep 2', 'Native Rep1 vs Rep2'),
            (native_13_100bp, stats_n13_100bp, 'Native Rep 1', 'Native Rep 3', 'Native Rep1 vs Rep3'),
            (native_23_100bp, stats_n23_100bp, 'Native Rep 2', 'Native Rep 3', 'Native Rep2 vs Rep3')
        ],
        'ivt_data': [
            (ivt_12_100bp, stats_i12_100bp, 'IVT Rep 1', 'IVT Rep 2', 'IVT Rep1 vs Rep2'),
            (ivt_13_100bp, stats_i13_100bp, 'IVT Rep 1', 'IVT Rep 3', 'IVT Rep1 vs Rep3'),
            (ivt_23_100bp, stats_i23_100bp, 'IVT Rep 2', 'IVT Rep 3', 'IVT Rep2 vs Rep3')
        ],
        'stats_list': [stats_n12_100bp, stats_n13_100bp, stats_n23_100bp,
                       stats_i12_100bp, stats_i13_100bp, stats_i23_100bp],
        'suffix': '_100bp',
        'label': '100bp bins'
    },
    'binned_1kb': {
        'native_data': [
            (native_12_1kb, stats_n12_1kb, 'Native Rep 1', 'Native Rep 2', 'Native Rep1 vs Rep2'),
            (native_13_1kb, stats_n13_1kb, 'Native Rep 1', 'Native Rep 3', 'Native Rep1 vs Rep3'),
            (native_23_1kb, stats_n23_1kb, 'Native Rep 2', 'Native Rep 3', 'Native Rep2 vs Rep3')
        ],
        'ivt_data': [
            (ivt_12_1kb, stats_i12_1kb, 'IVT Rep 1', 'IVT Rep 2', 'IVT Rep1 vs Rep2'),
            (ivt_13_1kb, stats_i13_1kb, 'IVT Rep 1', 'IVT Rep 3', 'IVT Rep1 vs Rep3'),
            (ivt_23_1kb, stats_i23_1kb, 'IVT Rep 2', 'IVT Rep 3', 'IVT Rep2 vs Rep3')
        ],
        'stats_list': [stats_n12_1kb, stats_n13_1kb, stats_n23_1kb,
                       stats_i12_1kb, stats_i13_1kb, stats_i23_1kb],
        'suffix': '_1kb',
        'label': '1kb bins'
    }
}

## Plots

In [ ]:
def plot_replicate_panel(ax, merged_df, name1, name2, stats_dict, title):
    cov1 = merged_df[f'cpm_{name1}'].values
    cov2 = merged_df[f'cpm_{name2}'].values

    # only positions with coverage in both samples
    mask = (cov1 > 0) & (cov2 > 0)
    cov1_log = np.log10(cov1[mask] + 1)
    cov2_log = np.log10(cov2[mask] + 1)

    hexbin = ax.hexbin(cov1_log, cov2_log, gridsize=50, cmap='Purples',
                       mincnt=1, bins='log')

    max_val = max(cov1_log.max(), cov2_log.max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, alpha=0.5,
            label='Perfect correlation')

    ax.set_xlabel(f'{name1.replace(" Rep ", " Rep")} coverage [log\u2081\u2080(CPM + 1)]', fontsize=16)
    ax.set_ylabel(f'{name2.replace(" Rep ", " Rep")} coverage [log\u2081\u2080(CPM + 1)]', fontsize=16)
    ax.set_title(title, fontsize=16, fontweight='bold')

    stats_text = (
        f"Positions: {stats_dict['n_positions']:,}\n"
        f"Spearman \u03c1: {stats_dict['spearman_r']:.3f}\n"
        f"Pearson r: {stats_dict['pearson_r']:.3f}"
    )
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
            fontsize=12, va='bottom', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

    ax.grid(True, alpha=0.3, linestyle='--')

    cbar = plt.colorbar(hexbin, ax=ax)
    cbar.set_label('Position count (log scale)', fontsize=11)

In [ ]:
for resolution, config in datasets.items():
    out_dir = Path('suppfig_s1') / resolution
    out_dir.mkdir(parents=True, exist_ok=True)

    suffix = config['suffix']
    label = config['label']
    native_data = config['native_data']
    ivt_data = config['ivt_data']

    fig = plt.figure(figsize=(18, 10))
    gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

    for idx, (merged_df, stats_dict, name1, name2, title) in enumerate(native_data):
        ax = fig.add_subplot(gs[0, idx])
        plot_replicate_panel(ax, merged_df, name1, name2, stats_dict, title)

    for idx, (merged_df, stats_dict, name1, name2, title) in enumerate(ivt_data):
        ax = fig.add_subplot(gs[1, idx])
        plot_replicate_panel(ax, merged_df, name1, name2, stats_dict, title)

    fig.suptitle('', fontsize=16, fontweight='bold')

    plt.savefig(out_dir / f'FigureS1_Replicate_Reproducibility{suffix}.png', dpi=300, bbox_inches='tight')
    plt.savefig(out_dir / f'FigureS1_Replicate_Reproducibility{suffix}.pdf', bbox_inches='tight')
    print(f"S1 ({label}): {out_dir}/FigureS1_Replicate_Reproducibility{suffix}.png/pdf")
    plt.close()

## Summary table

In [ ]:
all_stats = []
for resolution, config in datasets.items():
    for stats_dict in config['stats_list']:
        stats_copy = stats_dict.copy()
        stats_copy['resolution'] = config['label']
        stats_copy['Type'] = 'Native-Native' if 'Native' in stats_dict['comparison'] else 'IVT-IVT'
        all_stats.append(stats_copy)

summary_df = pd.DataFrame(all_stats)
summary_df['spearman_r'] = summary_df['spearman_r'].round(3)
summary_df['pearson_r'] = summary_df['pearson_r'].round(3)
summary_df = summary_df[['resolution', 'Type', 'comparison', 'n_positions', 'spearman_r', 'pearson_r',
                         'n_total', 'pct_covered_both', 'spearman_p', 'pearson_p']]

summary_df.to_csv('TableS1_Replicate_Correlations_all_resolutions.csv', index=False)
print(summary_df[['resolution', 'Type', 'comparison', 'n_positions', 'spearman_r', 'pearson_r']].to_string(index=False))

print("\nAverages by resolution:")
for resolution in ['Per-base', '100bp bins', '1kb bins']:
    res_df = summary_df[summary_df['resolution'] == resolution]
    native_df = res_df[res_df['Type'] == 'Native-Native']
    ivt_df = res_df[res_df['Type'] == 'IVT-IVT']
    print(f"  {resolution}:")
    print(f"    Native-Native: Spearman \u03c1 = {native_df['spearman_r'].mean():.3f}")
    print(f"    IVT-IVT:       Spearman \u03c1 = {ivt_df['spearman_r'].mean():.3f}")

# interpretation based on 1kb binned
res_1kb = summary_df[summary_df['resolution'] == '1kb bins']
native_1kb = res_1kb[res_1kb['Type'] == 'Native-Native']['spearman_r'].mean()
ivt_1kb = res_1kb[res_1kb['Type'] == 'IVT-IVT']['spearman_r'].mean()

if abs(native_1kb - ivt_1kb) < 0.05:
    print("\nIVT and Native correlations are similar — IVT process adds minimal noise.")
else:
    print(f"\nDifference of {abs(native_1kb - ivt_1kb):.3f} between Native and IVT correlations.")